TReX code 

In [ ]:
# ============================
# TReX: Temporal Regime Explainer Framework 
# COMPLETE VERSION WITH FIXED RANDOM SEEDS FOR CONSISTENT SHAP VALUES
# ============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import shap
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import networkx as nx
from scipy import stats
import joblib
import pickle
import random
import warnings
warnings.filterwarnings('ignore')

# ============================
# FIX RANDOM SEEDS FOR CONSISTENT SHAP VALUES (CORRECTED)
# ============================
def set_all_seeds(seed=42):
    """Fix all random seeds for reproducible SHAP values"""
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    
    # For SHAP's internal randomness - try different approaches
    try:
        shap.explainers._common.SEED = seed
    except AttributeError:
        try:
            shap.common.SEED = seed
        except AttributeError:
            pass  # SHAP version might not expose this attribute
    
    # Also set for sklearn
    from sklearn.utils import check_random_state
    _ = check_random_state(seed)

# Apply seeds immediately
set_all_seeds(42)

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# ============================
# CONFIGURATION - YOUR PATHS
# ============================

STATION_PATHS = {
    "Aljarafe": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Aljarafe Results\Aljarafe results and trained models",
    "Asomadilla": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Asomadilla Results\results_dmoff_consistent_metrics", 
    "Bermejales": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Bermejales Results\results_dmoff_consistent_metrics",
    "RonsadelVAalle": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\RonsadelVAalle Results\results_dmoff_consistent_metrics",
    "Torneo": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Torneo Results\results_dmoff_consistent_metrics"
}

DATASETS_PATHS = {
    "Aljarafe": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Aljarafe Results\Aljarafe.ods",
    "Bermejales": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Bermejales Results\Bermejales_2015_2023.ods",
    "Asomadilla": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Asomadilla Results\Asomadilla_2015_2023.ods",   
    "RonsadelVAalle": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\RonsadelVAalle Results\RonsadelVAalle_2015_2023.ods",
    "Torneo": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Torneo Results\Torneo_2015_2023.ods"
}

# ============================
# IMPROVED SHAP SAVING FUNCTION - WITH SEED FIXING
# ============================

def save_shap_values_for_ltrex(shap_values: dict, regime_labels: np.ndarray, 
                                feature_names: list, X_test: np.ndarray,
                                station_name: str, output_base_dir: str = "TReX_Publication_Results"):
    """
    Save COMPLETE SHAP information for L-TReX to use
    This ensures 100% consistency between frameworks
    """
    station_output_dir = os.path.join(output_base_dir, station_name)
    shap_dir = os.path.join(station_output_dir, "SHAP_Values_Complete")
    os.makedirs(shap_dir, exist_ok=True)
    
    # Convert any non-serializable objects
    serializable_shap_values = {}
    for regime, sv in shap_values.items():
        if hasattr(sv, 'tolist'):
            serializable_shap_values[regime] = sv.tolist()
        else:
            serializable_shap_values[regime] = sv
    
    # Save everything in a comprehensive dictionary
    complete_shap_data = {
        'shap_values': serializable_shap_values,
        'regime_labels': regime_labels.tolist() if hasattr(regime_labels, 'tolist') else regime_labels,
        'feature_names': feature_names,
        'X_test_shape': X_test.shape,
        'timestamp': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        'random_seed': 42  # Record the seed used
    }
    
    # Save as pickle
    with open(os.path.join(shap_dir, "complete_shap_data.pkl"), 'wb') as f:
        pickle.dump(complete_shap_data, f)
    
    # Also save each regime's SHAP values as separate files
    for regime, shap_vals in shap_values.items():
        regime_data = {
            'regime': regime,
            'shap_values': shap_vals.tolist() if hasattr(shap_vals, 'tolist') else shap_vals,
            'n_samples': shap_vals.shape[0] if hasattr(shap_vals, 'shape') else len(shap_vals),
            'n_features': shap_vals.shape[1] if hasattr(shap_vals, 'shape') and len(shap_vals.shape) > 1 else 1,
            'feature_names': feature_names,
            'random_seed': 42
        }
        with open(os.path.join(shap_dir, f"shap_data_regime_{regime}.pkl"), 'wb') as f:
            pickle.dump(regime_data, f)
        
        # Save as numpy for easy loading
        try:
            np.save(os.path.join(shap_dir, f"shap_values_regime_{regime}.npy"), np.array(shap_vals))
        except:
            print(f"   ⚠️ Could not save numpy array for regime {regime}")
    
    # Save feature importance summary as CSV
    importance_summary = []
    for regime, shap_vals in shap_values.items():
        shap_vals_array = np.array(shap_vals)
        if len(shap_vals_array.shape) == 3:
            mean_abs_shap = np.mean(np.abs(shap_vals_array), axis=(0, 1))
        else:
            mean_abs_shap = np.mean(np.abs(shap_vals_array), axis=0)
        
        for i, feat in enumerate(feature_names):
            if i < len(mean_abs_shap):
                importance_summary.append({
                    'Regime': regime,
                    'Feature': feat,
                    'Mean_Abs_SHAP': float(mean_abs_shap[i])
                })
    
    importance_df = pd.DataFrame(importance_summary)
    importance_df.to_csv(os.path.join(shap_dir, "feature_importance_summary.csv"), index=False)
    
    print(f"   ✅ Saved COMPLETE SHAP data for {len(shap_values)} regimes to {shap_dir}")
    print(f"   📊 Feature importance summary saved to CSV for verification")
    
    return shap_dir


# ============================
# COMPLETE TEMPORAL REGIME EXPLAINER WITH ALL ANALYSES
# ============================

class TemporalRegimeExplainer:
    """
    TReX FRAMEWORK WITH COMPLETE ANALYSES
    """
    
    def __init__(self, regime_models, regime_classifier, feature_names, temporal_window=24):
        self.regime_models = regime_models
        self.regime_classifier = regime_classifier
        self.feature_names = feature_names
        self.temporal_window = temporal_window
        
    def get_model_feature_names(self, regime):
        """Get the actual features used by a specific regime's model"""
        regime_data = self.regime_models.get(regime, {})
        
        if 'rf_model' in regime_data:
            model = regime_data['rf_model']
            if hasattr(model, 'feature_names_in_'):
                return list(model.feature_names_in_)
            elif hasattr(model, 'feature_importances_'):
                n_features = len(model.feature_importances_)
                return self.feature_names[:n_features]
        
        return self.feature_names
    
    def align_features(self, X, regime):
        """Align feature dimensions between data and model"""
        model_features = self.get_model_feature_names(regime)
        available_features = self.feature_names
        
        feature_indices = []
        for model_feat in model_features:
            if model_feat in available_features:
                feature_indices.append(available_features.index(model_feat))
        
        if len(feature_indices) == 0:
            return X[:, :len(model_features)]
        
        return X[:, feature_indices]
    
    def regime_aware_shap(self, X_test, regime_labels, n_background=100):
        """
        INNOVATION 1: Regime-Aware SHAP with fixed random seeds
        """
        print("Computing Regime-Aware SHAP explanations...")
        
        regime_shap_values = {}
        regime_explainers = {}
        
        for regime in np.unique(regime_labels):
            regime_mask = regime_labels == regime
            X_regime = X_test[regime_mask]
            
            if len(X_regime) < 10:
                continue
                
            X_regime_aligned = self.align_features(X_regime, regime)
            model = self._get_regime_model(regime)
            
            if model is not None:
                try:
                    # Use fixed seed for background selection
                    background_data = self._select_regime_background(regime, X_regime_aligned, n_background)
                    
                    if hasattr(model, 'estimators_') or hasattr(model, 'tree_'):
                        # Tree-based model (Random Forest, XGBoost, etc.)
                        explainer = shap.TreeExplainer(model)
                        shap_values = explainer.shap_values(X_regime_aligned)
                        # Handle multi-class output
                        if isinstance(shap_values, list):
                            shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
                    else:
                        # For other models, use KernelExplainer with fixed seed
                        np.random.seed(42)
                        explainer = shap.KernelExplainer(model.predict, background_data)
                        shap_values = explainer.shap_values(X_regime_aligned, nsamples=100)
                    
                    regime_shap_values[regime] = shap_values
                    regime_explainers[regime] = explainer
                    
                    shap_shape = shap_values.shape if hasattr(shap_values, 'shape') else 'complex'
                    print(f"   ✅ Regime {regime}: SHAP computed for {len(X_regime)} samples, shape={shap_shape}")
                    
                except Exception as e:
                    print(f"   ⚠️ SHAP failed for regime {regime}: {str(e)[:100]}...")
        
        return regime_shap_values, regime_explainers
    
    def _select_regime_background(self, regime, X_regime, n_background):
        """Smart background data selection with fixed random state"""
        if len(X_regime) <= n_background:
            return X_regime
        
        try:
            n_clusters = min(n_background, len(X_regime))
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=5)
            labels = kmeans.fit_predict(X_regime)
            
            background_indices = []
            for cluster_id in range(n_clusters):
                cluster_points = np.where(labels == cluster_id)[0]
                if len(cluster_points) > 0:
                    center = kmeans.cluster_centers_[cluster_id]
                    distances = np.linalg.norm(X_regime[cluster_points] - center, axis=1)
                    background_indices.append(cluster_points[np.argmin(distances)])
            
            return X_regime[background_indices]
        except:
            np.random.seed(42)
            indices = np.random.choice(len(X_regime), size=min(n_background, len(X_regime)), replace=False)
            return X_regime[indices]
    
    def _get_regime_model(self, regime):
        """Get the appropriate model for a given regime"""
        regime_data = self.regime_models.get(regime, {})
        
        if 'rf_model' in regime_data:
            return regime_data['rf_model']
        elif 'lgb_model' in regime_data:
            return regime_data['lgb_model']
        elif 'lstm_model' in regime_data:
            return self._create_lstm_predictor(regime_data['lstm_model'])
        else:
            return None
    
    def _create_lstm_predictor(self, lstm_model):
        """Create LSTM predictor wrapper"""
        def lstm_predictor(X):
            try:
                if len(X.shape) == 2:
                    X_reshaped = X.reshape(X.shape[0], self.temporal_window, -1)
                    return lstm_model.predict(X_reshaped, verbose=0).flatten()
                return lstm_model.predict(X, verbose=0).flatten()
            except:
                return np.array([0.5] * len(X))
        return lstm_predictor

    def create_shap_visualizations(self, shap_values, regime_labels, feature_names, station_name, output_dir):
        """
        CREATE PUBLICATION-READY SHAP VISUALIZATIONS
        """
        print("Creating SHAP visualizations...")
        
        os.makedirs(output_dir, exist_ok=True)
        
        # 1. Summary Plot for each regime
        for regime in np.unique(regime_labels):
            if regime not in shap_values:
                continue
                
            shap_vals = np.array(shap_values[regime])
            print(f"   Regime {regime}: SHAP shape = {shap_vals.shape}")
            
            if len(shap_vals.shape) == 3:
                shap_vals_2d = np.mean(shap_vals, axis=1)
            elif len(shap_vals.shape) == 2:
                shap_vals_2d = shap_vals
            else:
                shap_vals_2d = shap_vals.reshape(1, -1)
            
            # Ensure feature names match
            n_features = min(shap_vals_2d.shape[1] if len(shap_vals_2d.shape) > 1 else len(shap_vals_2d), len(feature_names))
            vis_feature_names = feature_names[:n_features]
            if len(shap_vals_2d.shape) > 1:
                shap_vals_2d = shap_vals_2d[:, :n_features]
            
            try:
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_vals_2d, feature_names=vis_feature_names, 
                                show=False, plot_size=None)
                plt.title(f'SHAP Summary Plot - {station_name} - Regime {regime}', fontsize=16, fontweight='bold')
                plt.tight_layout()
                plt.savefig(f"{output_dir}/SHAP_Summary_Regime_{regime}_{station_name}.png", dpi=300, bbox_inches='tight')
                plt.savefig(f"{output_dir}/SHAP_Summary_Regime_{regime}_{station_name}.pdf", bbox_inches='tight')
                plt.close()
            except Exception as e:
                print(f"   ⚠️ Could not create summary plot for regime {regime}: {e}")
            
            # Also save feature importance bar plot
            if len(shap_vals_2d.shape) > 1:
                mean_abs_shap = np.mean(np.abs(shap_vals_2d), axis=0)
            else:
                mean_abs_shap = np.abs(shap_vals_2d)
            
            sorted_idx = np.argsort(mean_abs_shap)[::-1]
            
            plt.figure(figsize=(12, 8))
            colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(sorted_idx)))
            plt.barh(range(len(sorted_idx)), mean_abs_shap[sorted_idx], color=colors, edgecolor='black')
            plt.yticks(range(len(sorted_idx)), [vis_feature_names[i] for i in sorted_idx])
            plt.xlabel('Mean |SHAP Value|', fontsize=12)
            plt.title(f'Feature Importance - {station_name} - Regime {regime}', fontsize=14, fontweight='bold')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.savefig(f"{output_dir}/Feature_Importance_Regime_{regime}_{station_name}.png", dpi=300, bbox_inches='tight')
            plt.close()
        
        print(f"   ✅ SHAP visualizations saved to {output_dir}")
    
    # ============================
    # ADDITIONAL ANALYSES FROM OLD TReX
    # ============================
    
    def temporal_influence_analysis(self, shap_values, regime_labels, feature_names, station_name, output_dir):
        """
        INNOVATION 2: Temporal Influence Analysis with Visualizations
        """
        print("Performing Temporal Influence Analysis...")
        
        os.makedirs(output_dir, exist_ok=True)
        
        # 1. Create temporal influence graphs
        temporal_graphs = self._create_temporal_graphs(shap_values, regime_labels, feature_names)
        
        # 2. Plot temporal influence networks
        self._plot_temporal_networks(temporal_graphs, station_name, output_dir)
        
        # 3. Create temporal influence table
        self._create_temporal_influence_table(shap_values, feature_names, station_name, output_dir)
        
        return temporal_graphs
    
    def _create_temporal_graphs(self, shap_values, regime_labels, feature_names):
        """Create temporal influence graphs"""
        temporal_graphs = {}
        
        for regime in np.unique(regime_labels):
            if regime not in shap_values:
                continue
                
            regime_shap = np.array(shap_values[regime])
            
            if len(regime_shap.shape) == 3:
                temporal_importance = np.mean(np.abs(regime_shap), axis=1)
            else:
                temporal_importance = np.abs(regime_shap)
            
            if temporal_importance.shape[1] > len(feature_names):
                temporal_importance = temporal_importance[:, :len(feature_names)]
            
            G = nx.Graph()
            
            for i, feature in enumerate(feature_names):
                if i < temporal_importance.shape[1]:
                    avg_importance = np.mean(temporal_importance[:, i])
                    G.add_node(feature, importance=float(avg_importance))
            
            if temporal_importance.shape[1] > 1:
                try:
                    importance_corr = np.corrcoef(temporal_importance.T)
                    
                    for i in range(min(len(feature_names), importance_corr.shape[0])):
                        for j in range(i+1, min(len(feature_names), importance_corr.shape[1])):
                            if i != j and abs(importance_corr[i, j]) > 0.2:
                                G.add_edge(feature_names[i], feature_names[j], 
                                         weight=float(abs(importance_corr[i, j])))
                except:
                    pass
            
            temporal_graphs[regime] = G
        
        return temporal_graphs
    
    def _plot_temporal_networks(self, temporal_graphs, station_name, output_dir):
        """Plot temporal influence networks"""
        for regime, graph in temporal_graphs.items():
            if graph.number_of_nodes() == 0:
                continue
                
            plt.figure(figsize=(15, 12))
            
            node_sizes = [3000 * graph.nodes[node].get('importance', 0.1) for node in graph.nodes()]
            edge_weights = [graph[u][v]['weight'] * 5 for u, v in graph.edges()]
            
            pos = nx.spring_layout(graph, k=2, iterations=50, seed=42)
            
            nx.draw_networkx_nodes(graph, pos, node_size=node_sizes, 
                                 node_color='lightcoral', alpha=0.8, edgecolors='black')
            nx.draw_networkx_edges(graph, pos, width=edge_weights, 
                                 alpha=0.6, edge_color='gray')
            nx.draw_networkx_labels(graph, pos, font_size=10, font_weight='bold')
            
            # Add edge labels
            edge_labels = {(u, v): f"{graph[u][v]['weight']:.2f}" for u, v in graph.edges()}
            nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8)
            
            plt.title(f'Temporal Influence Network - {station_name} - Regime {regime}', 
                     fontsize=16, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(f"{output_dir}/Temporal_Network_Regime_{regime}_{station_name}.png", 
                       dpi=300, bbox_inches='tight')
            plt.savefig(f"{output_dir}/Temporal_Network_Regime_{regime}_{station_name}.pdf", 
                       bbox_inches='tight')
            plt.close()
    
    def _create_temporal_influence_table(self, shap_values, feature_names, station_name, output_dir):
        """Create temporal influence table"""
        regimes = sorted(shap_values.keys())
        influence_data = []
        
        for regime in regimes:
            shap_vals = np.array(shap_values[regime])
            if len(shap_vals.shape) == 3:
                importance = np.mean(np.abs(shap_vals), axis=(0, 1))
            else:
                importance = np.mean(np.abs(shap_vals), axis=0)
            
            for i, feature in enumerate(feature_names):
                if i < len(importance):
                    influence_data.append({
                        'Regime': f'Regime {regime}',
                        'Feature': feature,
                        'Mean_Abs_SHAP': importance[i],
                        'Rank': i + 1
                    })
        
        if influence_data:
            influence_df = pd.DataFrame(influence_data)
            influence_df.to_csv(f"{output_dir}/Temporal_Influence_Table_{station_name}.csv", index=False)
            influence_df.to_excel(f"{output_dir}/Temporal_Influence_Table_{station_name}.xlsx", index=False)
    
    def regime_transition_analysis(self, X_test, regime_labels, station_name, output_dir):
        """
        INNOVATION 3: Regime Transition Analysis with Visualizations
        """
        print("Analyzing regime transitions...")
        
        os.makedirs(output_dir, exist_ok=True)
        
        transitions = []
        for i in range(1, len(regime_labels)):
            if regime_labels[i] != regime_labels[i-1]:
                transitions.append({
                    'index': i,
                    'from_regime': int(regime_labels[i-1]),
                    'to_regime': int(regime_labels[i])
                })
        
        print(f"   Found {len(transitions)} regime transitions")
        
        # Analyze transitions and create visualizations
        transition_results = self._analyze_transitions(X_test, regime_labels, transitions, station_name, output_dir)
        
        return transition_results
    
    def _analyze_transitions(self, X_test, regime_labels, transitions, station_name, output_dir):
        """Analyze regime transitions and create visualizations"""
        transition_analysis = {}
        
        for i, trans in enumerate(transitions[:5]):  # Analyze first 5 transitions
            start_idx = max(0, trans['index'] - 15)
            end_idx = min(len(X_test), trans['index'] + 15)
            
            if end_idx - start_idx < 10:
                continue
                
            X_window = X_test[start_idx:end_idx]
            regimes_window = regime_labels[start_idx:end_idx]
            
            analysis = self._analyze_single_transition(X_window, regimes_window, trans)
            transition_analysis[trans['index']] = analysis
            
            # Create transition visualization
            self._plot_transition(X_window, regimes_window, trans, station_name, output_dir, i)
        
        # Create transition summary table
        self._create_transition_summary_table(transition_analysis, station_name, output_dir)
        
        return transition_analysis
    
    def _analyze_single_transition(self, X_window, regimes_window, transition):
        """Analyze a single transition"""
        transition_idx = None
        for i in range(1, len(regimes_window)):
            if regimes_window[i] != regimes_window[i-1]:
                transition_idx = i
                break
        
        if transition_idx is None:
            return {"error": "No transition found"}
        
        pre_transition = X_window[:transition_idx]
        post_transition = X_window[transition_idx:]
        
        feature_changes = {}
        for i, feature in enumerate(self.feature_names):
            if i >= X_window.shape[1]:
                continue
                
            if len(pre_transition) > 2 and len(post_transition) > 2:
                try:
                    stat, p_value = stats.ks_2samp(pre_transition[:, i], post_transition[:, i])
                    mean_change = np.mean(post_transition[:, i]) - np.mean(pre_transition[:, i])
                    std_pooled = np.std(np.concatenate([pre_transition[:, i], post_transition[:, i]]))
                    effect_size = abs(mean_change) / (std_pooled + 1e-8)
                    
                    feature_changes[feature] = {
                        'ks_statistic': float(stat),
                        'p_value': float(p_value),
                        'mean_change': float(mean_change),
                        'effect_size': float(effect_size),
                        'significant': p_value < 0.05 and effect_size > 0.3
                    }
                except:
                    continue
        
        driving_features = [
            feat for feat, analysis in feature_changes.items() 
            if analysis.get('significant', False)
        ]
        
        return {
            'transition_point': transition_idx,
            'pre_regime': int(transition['from_regime']),
            'post_regime': int(transition['to_regime']),
            'feature_changes': feature_changes,
            'driving_features': driving_features
        }
    
    def _plot_transition(self, X_window, regimes_window, transition, station_name, output_dir, trans_num):
        """Plot a single transition"""
        transition_idx = None
        for i in range(1, len(regimes_window)):
            if regimes_window[i] != regimes_window[i-1]:
                transition_idx = i
                break
        
        if transition_idx is None:
            return
        
        # Plot top 5 features with largest changes
        feature_changes = {}
        for i, feature in enumerate(self.feature_names):
            if i >= X_window.shape[1]:
                continue
                
            pre_mean = np.mean(X_window[:transition_idx, i])
            post_mean = np.mean(X_window[transition_idx:, i])
            change = abs(post_mean - pre_mean)
            feature_changes[feature] = change
        
        # Get top 5 features with largest changes
        top_features = sorted(feature_changes.items(), key=lambda x: x[1], reverse=True)[:5]
        
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        time_points = np.arange(len(X_window))
        
        for idx, (feature, change) in enumerate(top_features):
            if idx >= len(axes) - 1:
                break
                
            feature_idx = self.feature_names.index(feature)
            axes[idx].plot(time_points, X_window[:, feature_idx], 'b-', alpha=0.7, linewidth=2)
            axes[idx].axvline(x=transition_idx, color='red', linestyle='--', linewidth=2, 
                            label=f'Transition (Regime {transition["from_regime"]} → {transition["to_regime"]})')
            axes[idx].set_title(f'{feature}\nChange: {change:.3f}', fontweight='bold')
            axes[idx].set_xlabel('Time Step')
            axes[idx].set_ylabel('Value')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)
        
        # Add regime plot in the last subplot
        axes[-1].plot(time_points, regimes_window, 'g-', linewidth=3)
        axes[-1].axvline(x=transition_idx, color='red', linestyle='--', linewidth=2)
        axes[-1].set_title('Regime Changes', fontweight='bold')
        axes[-1].set_xlabel('Time Step')
        axes[-1].set_ylabel('Regime')
        axes[-1].grid(True, alpha=0.3)
        
        plt.suptitle(f'Regime Transition Analysis - {station_name} - Transition {trans_num+1}', 
                    fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{output_dir}/Transition_{trans_num+1}_{station_name}.png", dpi=300, bbox_inches='tight')
        plt.savefig(f"{output_dir}/Transition_{trans_num+1}_{station_name}.pdf", bbox_inches='tight')
        plt.close()
    
    def _create_transition_summary_table(self, transition_analysis, station_name, output_dir):
        """Create transition summary table"""
        summary_data = []
        
        for trans_point, analysis in transition_analysis.items():
            driving_features = analysis.get('driving_features', [])
            summary_data.append({
                'Transition_Point': trans_point,
                'From_Regime': analysis.get('pre_regime', 'N/A'),
                'To_Regime': analysis.get('post_regime', 'N/A'),
                'Driving_Features': ', '.join(driving_features) if driving_features else 'None',
                'N_Driving_Features': len(driving_features)
            })
        
        if summary_data:
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_csv(f"{output_dir}/Transition_Summary_{station_name}.csv", index=False)
            summary_df.to_excel(f"{output_dir}/Transition_Summary_{station_name}.xlsx", index=False)
    
    def cross_regime_consistency_analysis(self, shap_values, regime_labels, station_name, output_dir):
        """
        INNOVATION 4: Cross-Regime Consistency Analysis
        """
        print("Performing Cross-Regime Consistency Analysis...")
        
        os.makedirs(output_dir, exist_ok=True)
        
        regimes = sorted([r for r in np.unique(regime_labels) if r in shap_values])
        
        # Compute feature importance for each regime
        regime_importances = {}
        for regime in regimes:
            shap_vals = np.array(shap_values[regime])
            if len(shap_vals.shape) == 3:
                importance = np.mean(np.abs(shap_vals), axis=(0, 1))
            else:
                importance = np.mean(np.abs(shap_vals), axis=0)
            
            valid_importance = np.zeros(len(self.feature_names))
            for i in range(min(len(importance), len(self.feature_names))):
                valid_importance[i] = importance[i]
            
            regime_importances[regime] = valid_importance
        
        # Compute consistency metrics
        consistency_results = {}
        for i, feature in enumerate(self.feature_names):
            importances = []
            for regime in regimes:
                if i < len(regime_importances[regime]):
                    imp = regime_importances[regime][i]
                    if not np.isnan(imp):
                        importances.append(imp)
            
            if importances:
                mean_imp = np.mean(importances)
                std_imp = np.std(importances)
                cv = std_imp / (mean_imp + 1e-8)
                
                consistency_results[feature] = {
                    'mean_importance': float(mean_imp),
                    'std_importance': float(std_imp),
                    'cv_importance': float(cv),
                    'consistency': 'High' if cv < 0.5 else 'Medium' if cv < 1.0 else 'Low',
                    'n_regimes': len(importances)
                }
        
        # Create consistency visualization
        self._plot_consistency_analysis(consistency_results, station_name, output_dir)
        
        # Save consistency table
        consistency_df = pd.DataFrame.from_dict(consistency_results, orient='index')
        consistency_df.index.name = 'Feature'
        consistency_df.reset_index(inplace=True)
        consistency_df.to_csv(f"{output_dir}/Consistency_Analysis_{station_name}.csv", index=False)
        consistency_df.to_excel(f"{output_dir}/Consistency_Analysis_{station_name}.xlsx", index=False)
        
        return consistency_results
    
    def _plot_consistency_analysis(self, consistency_results, station_name, output_dir):
        """Plot consistency analysis results"""
        if not consistency_results:
            return
            
        features = list(consistency_results.keys())
        cv_values = [consistency_results[f]['cv_importance'] for f in features]
        mean_importance = [consistency_results[f]['mean_importance'] for f in features]
        consistency_categories = [consistency_results[f]['consistency'] for f in features]
        
        # Create scatter plot
        plt.figure(figsize=(12, 8))
        
        colors = {'High': 'green', 'Medium': 'orange', 'Low': 'red'}
        color_list = [colors[cat] for cat in consistency_categories]
        
        plt.scatter(mean_importance, cv_values, c=color_list, s=100, alpha=0.7, edgecolors='black')
        
        # Add labels for top features
        for i, feature in enumerate(features):
            if mean_importance[i] > np.percentile(mean_importance, 75):
                plt.annotate(feature, (mean_importance[i], cv_values[i]), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        plt.axhline(y=0.5, color='green', linestyle='--', alpha=0.7, label='High Consistency Threshold')
        plt.axhline(y=1.0, color='orange', linestyle='--', alpha=0.7, label='Medium Consistency Threshold')
        
        plt.xlabel('Mean Feature Importance', fontsize=12, fontweight='bold')
        plt.ylabel('Coefficient of Variation (CV)', fontsize=12, fontweight='bold')
        plt.title(f'Cross-Regime Consistency Analysis - {station_name}', fontsize=14, fontweight='bold')
        
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='green', 
                                markersize=10, label='High Consistency (CV < 0.5)'),
                         Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', 
                                markersize=10, label='Medium Consistency (0.5 ≤ CV < 1.0)'),
                         Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                                markersize=10, label='Low Consistency (CV ≥ 1.0)')]
        
        plt.legend(handles=legend_elements, loc='upper right')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{output_dir}/Consistency_Analysis_{station_name}.png", dpi=300, bbox_inches='tight')
        plt.savefig(f"{output_dir}/Consistency_Analysis_{station_name}.pdf", bbox_inches='tight')
        plt.close()
    
    def generate_comprehensive_report(self, all_results, station_name, output_dir):
        """Generate comprehensive report with all visualizations and tables"""
        print("Generating comprehensive report...")
        
        report_data = {
            'station': station_name,
            'timestamp': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
            'framework': 'TReX v3.0 - Publication Ready',
            'random_seed': 42,
            'novel_contributions': [
                'Regime-Aware SHAP with Feature Alignment',
                'Temporal Influence Graphs',
                'Regime Transition Analysis',
                'Cross-Regime Consistency Analysis'
            ],
            'summary': {
                'n_regimes_analyzed': len(all_results.get('shap_values', {})),
                'n_transitions_analyzed': len(all_results.get('regime_transitions', {})),
                'n_features': len(self.feature_names)
            }
        }
        
        # Save report as JSON and text file
        import json
        with open(f"{output_dir}/TReX_Report_{station_name}.json", 'w') as f:
            json.dump(report_data, f, indent=2)
        
        # Create a readable text report
        with open(f"{output_dir}/TReX_Report_{station_name}.txt", 'w') as f:
            f.write("TReX FRAMEWORK - COMPREHENSIVE REPORT\n")
            f.write("=" * 50 + "\n\n")
            f.write(f"Station: {station_name}\n")
            f.write(f"Analysis Date: {report_data['timestamp']}\n")
            f.write(f"Framework: {report_data['framework']}\n")
            f.write(f"Random Seed: {report_data['random_seed']}\n\n")
            
            f.write("NOVEL CONTRIBUTIONS:\n")
            for contribution in report_data['novel_contributions']:
                f.write(f"  • {contribution}\n")
            
            f.write(f"\nSUMMARY:\n")
            f.write(f"  • Regimes Analyzed: {report_data['summary']['n_regimes_analyzed']}\n")
            f.write(f"  • Transitions Analyzed: {report_data['summary']['n_transitions_analyzed']}\n")
            f.write(f"  • Features: {report_data['summary']['n_features']}\n")
        
        print(f"   ✅ Comprehensive report generated for {station_name}")


# ============================
# DATA LOADING
# ============================

def load_station_data(station_name, results_path, dataset_path):
    """Robust data loading with error handling"""
    print(f"📂 Loading data for {station_name}...")
    
    try:
        # Load test predictions
        test_predictions_path = os.path.join(results_path, "test_predictions_all_models.csv")
        if not os.path.exists(test_predictions_path):
            test_predictions_path = os.path.join(results_path, "test_predictions_dmoff.csv")
        
        test_df = pd.read_csv(test_predictions_path)
        print(f"    ✅ Loaded test data: {len(test_df)} samples")
        
        # Load regime models
        regime_models_path = os.path.join(results_path, "regime_models.pkl")
        regime_models = joblib.load(regime_models_path)
        print(f"    ✅ Loaded regime models for {len(regime_models)} regimes")
        
        # Load regime classifier
        classifier_path = os.path.join(results_path, "regime_classifier_rf.pkl")
        regime_classifier = joblib.load(classifier_path)
        print(f"    ✅ Loaded regime classifier")
        
        # Extract feature names safely
        feature_names = extract_feature_names_safe(test_df)
        print(f"    ✅ Identified {len(feature_names)} features")
        
        return test_df, regime_models, regime_classifier, feature_names
        
    except Exception as e:
        print(f"    ❌ Error loading {station_name}: {e}")
        return None, None, None, None

def extract_feature_names_safe(test_df):
    """Safe feature name extraction"""
    exclude_cols = {'FECHA_HORA', 'regime', 'pred_regime', 'obs', 'pred_dmoff', 
                   'pred_rf_only', 'pred_lgb_only', 'pred_lstm_only', 'index', 'level_0', 'Unnamed: 0'}
    
    all_cols = set(test_df.columns)
    feature_candidates = all_cols - exclude_cols
    
    numeric_features = []
    for col in feature_candidates:
        if col in test_df.columns and pd.api.types.is_numeric_dtype(test_df[col]):
            if test_df[col].nunique() > 1:
                numeric_features.append(col)
    
    return numeric_features


# ============================
# CROSS-STATION ANALYSIS
# ============================

def generate_cross_station_analysis(all_station_results, successful_stations):
    """Generate cross-station comparative analysis"""
    cross_output_dir = "TReX_Publication_Results/Cross_Station_Analysis"
    os.makedirs(cross_output_dir, exist_ok=True)
    
    # Create station comparison table
    comparison_data = []
    for station_name, results in all_station_results.items():
        if 'results' in results and 'cross_regime_consistency' in results['results']:
            consistency_data = results['results']['cross_regime_consistency']
            
            # Count consistency categories
            high_consistency = sum(1 for data in consistency_data.values() if data['consistency'] == 'High')
            medium_consistency = sum(1 for data in consistency_data.values() if data['consistency'] == 'Medium')
            low_consistency = sum(1 for data in consistency_data.values() if data['consistency'] == 'Low')
            
            comparison_data.append({
                'Station': station_name,
                'Total_Features': len(consistency_data),
                'High_Consistency_Features': high_consistency,
                'Medium_Consistency_Features': medium_consistency,
                'Low_Consistency_Features': low_consistency,
                'High_Consistency_Percentage': (high_consistency / len(consistency_data)) * 100 if len(consistency_data) > 0 else 0
            })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        comparison_df.to_csv(f"{cross_output_dir}/Cross_Station_Comparison.csv", index=False)
        comparison_df.to_excel(f"{cross_output_dir}/Cross_Station_Comparison.xlsx", index=False)
        
        # Create comparison visualization
        plt.figure(figsize=(12, 8))
        stations = comparison_df['Station']
        high_pct = comparison_df['High_Consistency_Percentage']
        
        plt.bar(stations, high_pct, color='green', alpha=0.7, edgecolor='black')
        plt.ylabel('High Consistency Features (%)', fontweight='bold')
        plt.xlabel('Station', fontweight='bold')
        plt.title('Cross-Station Consistency Comparison', fontsize=16, fontweight='bold')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{cross_output_dir}/Cross_Station_Consistency.png", dpi=300, bbox_inches='tight')
        plt.savefig(f"{cross_output_dir}/Cross_Station_Consistency.pdf", bbox_inches='tight')
        plt.close()
        
        print(f"   ✅ Cross-station analysis saved to {cross_output_dir}")


# ============================
# MAIN EXECUTION PIPELINE
# ============================

def run_trex_for_station(station_name, results_path, dataset_path, output_base_dir="TReX_Publication_Results"):
    """Complete TReX analysis for one station with all analyses"""
    
    # Create organized output directory structure
    station_output_dir = os.path.join(output_base_dir, station_name)
    visualization_dir = os.path.join(station_output_dir, "Visualizations")
    tables_dir = os.path.join(station_output_dir, "Tables")
    reports_dir = os.path.join(station_output_dir, "Reports")
    shap_dir = os.path.join(station_output_dir, "SHAP_Values_Complete")
    
    for directory in [station_output_dir, visualization_dir, tables_dir, reports_dir, shap_dir]:
        os.makedirs(directory, exist_ok=True)
    
    # Load data
    test_df, regime_models, regime_classifier, feature_names = load_station_data(
        station_name, results_path, dataset_path
    )
    
    if test_df is None:
        return None
    
    # Prepare data
    X_test = test_df[feature_names].values
    regime_labels = test_df['regime'].values
    
    print(f"\n📊 Data: {X_test.shape} samples, {len(feature_names)} features")
    print(f"📊 Regimes: {np.unique(regime_labels)}")
    
    # Initialize TReX
    trex = TemporalRegimeExplainer(
        regime_models=regime_models,
        regime_classifier=regime_classifier,
        feature_names=feature_names
    )
    
    # Run all analyses
    all_results = {}
    
    try:
        # 1. Regime-Aware SHAP with Visualizations
        print("\n🔬 Regime-Aware SHAP Analysis...")
        shap_values, explainers = trex.regime_aware_shap(X_test, regime_labels, n_background=50)
        all_results['shap_values'] = shap_values
        
        # Save COMPLETE SHAP data for L-TReX
        print("\n💾 Saving SHAP data for L-TReX...")
        save_shap_values_for_ltrex(shap_values, regime_labels, feature_names, X_test, station_name, output_base_dir)
        
        # Create SHAP visualizations
        trex.create_shap_visualizations(shap_values, regime_labels, feature_names, station_name, visualization_dir)
        
        # 2. Temporal Influence Analysis
        print("\n⏰ Temporal Influence Analysis...")
        temporal_graphs = trex.temporal_influence_analysis(shap_values, regime_labels, feature_names, station_name, visualization_dir)
        all_results['temporal_graphs'] = temporal_graphs
        
        # 3. Regime Transition Analysis
        print("\n🔄 Regime Transition Analysis...")
        transition_results = trex.regime_transition_analysis(X_test, regime_labels, station_name, visualization_dir)
        all_results['regime_transitions'] = transition_results
        
        # 4. Cross-Regime Consistency Analysis
        print("\n📊 Cross-Regime Consistency Analysis...")
        consistency_results = trex.cross_regime_consistency_analysis(shap_values, regime_labels, station_name, visualization_dir)
        all_results['cross_regime_consistency'] = consistency_results
        
    except Exception as e:
        print(f"❌ Analysis error for {station_name}: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # Generate comprehensive report
    trex.generate_comprehensive_report(all_results, station_name, reports_dir)
    
    print(f"\n✅ TReX analysis complete for {station_name}")
    print(f"   Results saved to: {station_output_dir}")
    
    return {
        'station': station_name,
        'results': all_results,
        'output_directory': station_output_dir
    }


def run_complete_trex_analysis():
    """Run TReX for all stations and generate cross-station analysis"""
    print("="*70)
    print("🚀 TReX FRAMEWORK - PUBLICATION READY")
    print("   WITH FIXED RANDOM SEEDS FOR CONSISTENT SHAP VALUES")
    print("="*70)
    
    all_station_results = {}
    successful_stations = []
    
    for station_name, results_path in STATION_PATHS.items():
        dataset_path = DATASETS_PATHS[station_name]
        
        if os.path.exists(results_path):
            print(f"\n{'='*60}")
            print(f"📌 Processing {station_name}...")
            print('='*60)
            station_results = run_trex_for_station(station_name, results_path, dataset_path)
            
            if station_results is not None:
                all_station_results[station_name] = station_results
                successful_stations.append(station_name)
                print(f"✅ {station_name}: SUCCESS")
            else:
                print(f"❌ {station_name}: FAILED")
        else:
            print(f"❌ {station_name}: Path not found - {results_path}")
    
    # Generate cross-station comparative analysis
    if successful_stations:
        print(f"\n{'='*60}")
        print("📊 GENERATING CROSS-STATION COMPARATIVE ANALYSIS...")
        print('='*60)
        generate_cross_station_analysis(all_station_results, successful_stations)
    
    # Final summary
    print(f"\n{'='*60}")
    print(f"✅ TReX ANALYSIS COMPLETED!")
    print(f"Successful stations: {len(successful_stations)}/{len(STATION_PATHS)}")
    if successful_stations:
        print(f"Stations processed: {successful_stations}")
    print('='*60)
    
    return all_station_results


if __name__ == "__main__":
    results = run_complete_trex_analysis()

L-TReX code 

In [ ]:
# ============================================================================
# L-TReX FRAMEWORK - COMPLETE VERSION
# LOADS SHAP VALUES FROM TReX FOR 100% CONSISTENCY
# FIXED: Handles both list and numpy array formats
# ============================================================================

import os
import re
import json
import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from datetime import datetime
import shap
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# FIX RANDOM SEEDS FOR CONSISTENCY
# ============================================================================
def set_all_seeds(seed=42):
    np.random.seed(seed)
    import random
    random.seed(seed)
    
set_all_seeds(42)

# ============================================================================
# GROQ LLM API - FREE
# ============================================================================
try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    print("WARNING: Groq not installed. Run: pip install groq")

GROQ_API_KEY = "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"
GROQ_MODEL = "llama-3.1-8b-instant"

# ============================================================================
# HELPER FUNCTION: Convert SHAP values to numpy array safely
# ============================================================================

def to_numpy_array(shap_val):
    """Safely convert SHAP values to numpy array regardless of input format"""
    if isinstance(shap_val, np.ndarray):
        return shap_val
    elif isinstance(shap_val, list):
        # Convert nested list to numpy array
        return np.array(shap_val)
    elif hasattr(shap_val, 'shape'):
        return shap_val
    else:
        return np.array(shap_val)

def get_shap_shape(shap_val):
    """Safely get shape of SHAP values"""
    arr = to_numpy_array(shap_val)
    return arr.shape

# ============================================================================
# LOAD SHAP VALUES FROM TReX - THE SAME ONES TReX USED
# ============================================================================

def load_trex_complete_shap_data(station_name: str):
    """
    Load the COMPLETE SHAP data saved by TReX
    This ensures L-TReX uses EXACTLY the same SHAP values
    """
    print(f"\n{'='*60}")
    print(f"Loading SHAP data from TReX for {station_name}")
    print('='*60)
    
    # TReX saves in this directory
    trex_shap_dir = os.path.join("TReX_Publication_Results", station_name, "SHAP_Values_Complete")
    
    # Check for the complete pickle file
    complete_file = os.path.join(trex_shap_dir, "complete_shap_data.pkl")
    
    if os.path.exists(complete_file):
        print(f"✅ Found TReX SHAP data at: {trex_shap_dir}")
        
        with open(complete_file, 'rb') as f:
            complete_data = pickle.load(f)
        
        shap_values_raw = complete_data['shap_values']
        feature_names = complete_data['feature_names']
        regime_labels = complete_data['regime_labels']
        
        # Convert all SHAP values to numpy arrays
        shap_values = {}
        for regime, sv in shap_values_raw.items():
            shap_values[regime] = to_numpy_array(sv)
            print(f"      Regime {regime}: SHAP shape = {shap_values[regime].shape}")
        
        print(f"   ✅ Loaded SHAP values for {len(shap_values)} regimes")
        print(f"   ✅ Loaded {len(feature_names)} feature names")
        
        # Load feature importance summary for verification
        importance_csv = os.path.join(trex_shap_dir, "feature_importance_summary.csv")
        if os.path.exists(importance_csv):
            importance_df = pd.read_csv(importance_csv)
            print(f"   ✅ Loaded feature importance summary")
            
            # Show top features for regime 0 to verify
            if 0 in importance_df['Regime'].values:
                regime0_importance = importance_df[importance_df['Regime'] == 0].sort_values('Mean_Abs_SHAP', ascending=False)
                print(f"\n   📊 Top 5 features for Regime 0 (from TReX):")
                for i in range(min(5, len(regime0_importance))):
                    row = regime0_importance.iloc[i]
                    print(f"      {i+1}. {row['Feature']}: {row['Mean_Abs_SHAP']:.4f}")
        
        return shap_values, feature_names, regime_labels
    
    else:
        print(f"❌ TReX SHAP data NOT FOUND at: {trex_shap_dir}")
        print(f"   Please run the corrected TReX code first to generate SHAP values.")
        print(f"   Make sure TReX runs successfully before running L-TReX.")
        return None, None, None


# ============================================================================
# CREATE IDENTICAL SHAP VISUALIZATIONS TO TReX
# ============================================================================

def create_identical_shap_visualizations(shap_values: Dict, feature_names: List[str], 
                                           station_name: str, output_dir: str):
    """
    Creates SHAP visualizations that are EXACTLY IDENTICAL to TReX's output
    """
    print("\n" + "="*60)
    print("📊 Creating SHAP Visualizations (IDENTICAL to TReX)")
    print("="*60)
    
    shap_vis_dir = os.path.join(output_dir, "SHAP_Visualizations")
    os.makedirs(shap_vis_dir, exist_ok=True)
    
    for regime in sorted(shap_values.keys()):
        # Convert to numpy array safely
        shap_vals = to_numpy_array(shap_values[regime])
        
        print(f"\n   📈 Processing Regime {regime}...")
        print(f"      Original SHAP shape: {shap_vals.shape}")
        
        # CRITICAL: EXACT same processing as TReX
        if len(shap_vals.shape) == 3:
            shap_vals_2d = np.mean(shap_vals, axis=1)
        elif len(shap_vals.shape) == 2:
            shap_vals_2d = shap_vals
        else:
            shap_vals_2d = shap_vals.reshape(1, -1)
        
        print(f"      After processing shape: {shap_vals_2d.shape}")
        
        # Ensure feature names match
        n_features = min(shap_vals_2d.shape[1], len(feature_names))
        vis_feature_names = feature_names[:n_features]
        shap_vals_2d = shap_vals_2d[:, :n_features]
        
        # ========== 1. SHAP SUMMARY PLOT (IDENTICAL to TReX) ==========
        try:
            plt.figure(figsize=(12, 8))
            shap.summary_plot(shap_vals_2d, feature_names=vis_feature_names, 
                             show=False, plot_size=None)
            plt.title(f'SHAP Summary Plot - {station_name} - Regime {regime}', 
                     fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(shap_vis_dir, f'SHAP_Summary_Regime_{regime}_{station_name}.png'), 
                       dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(shap_vis_dir, f'SHAP_Summary_Regime_{regime}_{station_name}.pdf'), 
                       bbox_inches='tight')
            plt.close()
            print(f"      ✅ SHAP summary plot saved")
        except Exception as e:
            print(f"      ⚠️ Could not create summary plot: {e}")
        
        # ========== 2. FEATURE IMPORTANCE BAR PLOT ==========
        mean_abs_shap = np.mean(np.abs(shap_vals_2d), axis=0)
        sorted_idx = np.argsort(mean_abs_shap)[::-1]
        
        plt.figure(figsize=(12, 8))
        colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(sorted_idx)))
        plt.barh(range(len(sorted_idx)), mean_abs_shap[sorted_idx], color=colors, edgecolor='black')
        plt.yticks(range(len(sorted_idx)), [vis_feature_names[i] for i in sorted_idx])
        plt.xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
        plt.ylabel('Features', fontsize=12, fontweight='bold')
        plt.title(f'Feature Importance (Mean |SHAP|) - {station_name} - Regime {regime}', 
                 fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(os.path.join(shap_vis_dir, f'Feature_Importance_Regime_{regime}_{station_name}.png'), 
                   dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(shap_vis_dir, f'Feature_Importance_Regime_{regime}_{station_name}.pdf'), 
                   bbox_inches='tight')
        plt.close()
        
        # Save importance values as CSV
        importance_df = pd.DataFrame({
            'Feature': vis_feature_names,
            'Mean_Abs_SHAP': mean_abs_shap,
            'Rank': range(1, len(vis_feature_names) + 1)
        }).sort_values('Mean_Abs_SHAP', ascending=False)
        
        importance_df.to_csv(os.path.join(shap_vis_dir, f'Feature_Importance_Regime_{regime}_{station_name}.csv'), 
                            index=False)
        
        print(f"      ✅ Top 5 features for Regime {regime}:")
        for i in range(min(5, len(importance_df))):
            print(f"         {i+1}. {importance_df.iloc[i]['Feature']}: {importance_df.iloc[i]['Mean_Abs_SHAP']:.4f}")
    
    print(f"\n   ✅ All SHAP visualizations saved to: {shap_vis_dir}")
    print(f"   📁 These should be IDENTICAL to TReX's visualizations")
    
    return shap_vis_dir


# ============================================================================
# L-TReX Classes
# ============================================================================

@dataclass
class RegimeExplanationPacket:
    regime_id: str
    regime_name: str
    regime_description: str
    base_ozone_value: float
    base_ozone_unit: str = "ug/m3"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    top_features: List[Dict[str, Any]] = field(default_factory=list)
    counterfactual_scenarios: List[Dict[str, Any]] = field(default_factory=list)
    meteorological_conditions: Dict[str, float] = field(default_factory=dict)
    precursor_levels: Dict[str, float] = field(default_factory=dict)
    sample_size: int = 0
    temporal_context: str = ""


class LTRexDataAggregator:
    """Uses SHAP values from TReX to create explanation packets"""
    
    def __init__(self, shap_values: Dict, feature_names: List[str]):
        self.shap_values = shap_values
        self.feature_names = feature_names
        print(f"   📋 Initialized with {len(feature_names)} feature names")
    
    def extract_regime_packets(self, n_top_features: int = 5) -> List[RegimeExplanationPacket]:
        packets = []
        
        for regime_id, shap_vals_raw in self.shap_values.items():
            # Convert to numpy array safely
            shap_vals = to_numpy_array(shap_vals_raw)
            
            # Process SHAP values the SAME way as TReX
            if len(shap_vals.shape) == 3:
                shap_vals_2d = np.mean(shap_vals, axis=1)
            elif len(shap_vals.shape) == 2:
                shap_vals_2d = shap_vals
            else:
                shap_vals_2d = shap_vals.reshape(1, -1)
            
            # Extract top features
            if len(shap_vals_2d.shape) > 1:
                mean_shap_magnitude = np.mean(np.abs(shap_vals_2d), axis=0)
                mean_shap_signed = np.mean(shap_vals_2d, axis=0)
            else:
                mean_shap_magnitude = np.abs(shap_vals_2d)
                mean_shap_signed = shap_vals_2d
            
            n_features = min(len(mean_shap_magnitude), len(self.feature_names))
            
            feature_importance = []
            for i in range(n_features):
                feature_importance.append({
                    'name': self.feature_names[i],
                    'shap_magnitude': float(mean_shap_magnitude[i]),
                    'shap_value': float(mean_shap_signed[i]),
                    'effect': 'increase' if mean_shap_signed[i] > 0 else 'decrease'
                })
            
            feature_importance.sort(key=lambda x: x['shap_magnitude'], reverse=True)
            
            top_features = []
            for idx, fi in enumerate(feature_importance[:n_top_features]):
                top_features.append({
                    'name': fi['name'],
                    'shap_value': fi['shap_value'],
                    'shap_magnitude': fi['shap_magnitude'],
                    'effect': fi['effect'],
                    'rank': idx + 1
                })
            
            # Base ozone values (adjust based on your actual data)
            base_ozone_values = {0: 95.5, 1: 56.6, 2: 55.3, 3: 44.9, 4: 41.7}
            
            packet = RegimeExplanationPacket(
                regime_id=str(regime_id),
                regime_name=f"Regime {regime_id}",
                regime_description=self._get_regime_description(regime_id),
                base_ozone_value=base_ozone_values.get(regime_id, 50.0),
                top_features=top_features,
                counterfactual_scenarios=self._get_counterfactuals(regime_id),
                meteorological_conditions={},
                precursor_levels={'NO2': 30, 'PM10': 25},
                sample_size=shap_vals_2d.shape[0] if len(shap_vals_2d.shape) > 1 else 1,
                temporal_context=self._get_temporal_context(regime_id)
            )
            packets.append(packet)
            
            print(f"      📊 Regime {regime_id} top features: {[f['name'] for f in top_features[:3]]}")
        
        return packets
    
    def _get_regime_description(self, regime_id: int) -> str:
        descriptions = {
            0: "Cool regime - Moderate NO2, low wind speed, low temperature",
            1: "Warm regime - Higher temperature, lower NO2, weak winds",
            2: "Ventilated regime - Reduced pollution dominance, meteorological drivers gain relevance",
            3: "Moderate regime - Distributed influence among multiple predictors",
            4: "Highly polluted stable-air regime - NO2 dominance, suppressed photochemistry"
        }
        return descriptions.get(regime_id, f"Meteorological Regime {regime_id}")
    
    def _get_temporal_context(self, regime_id: int) -> str:
        contexts = {
            0: "Cool conditions with limited photochemical efficiency",
            1: "Warmer conditions with enhanced photochemistry",
            2: "Well-ventilated conditions",
            3: "Transitional conditions",
            4: "Stable, high-pollution conditions"
        }
        return contexts.get(regime_id, "Transitional seasonal conditions")
    
    def _get_counterfactuals(self, regime_id: int) -> List[Dict]:
        counterfactuals = [
            {'scenario_description': '20% Traffic Reduction', 'percent_change': -12.5, 'baseline_o3': 52.3, 'counterfactual_o3': 45.8},
            {'scenario_description': '30% Industrial Controls', 'percent_change': -18.2, 'baseline_o3': 52.3, 'counterfactual_o3': 42.8},
            {'scenario_description': 'Clean Energy Transition', 'percent_change': -22.5, 'baseline_o3': 52.3, 'counterfactual_o3': 40.5}
        ]
        return counterfactuals


class LTRexNarrativeGenerator:
    """Generates natural language explanations using FREE Groq LLM"""
    
    SYSTEM_PROMPT = """You are an Expert Environmental Policy Consultant specializing in air quality management and ozone pollution. Your role is to translate complex machine learning model explanations into clear, actionable policy narratives.

Your outputs must follow this EXACT 3-sentence structure:
- Sentence 1: Describe the current meteorological regime and its primary driver
- Sentence 2: Explain the impact of specific precursor emissions on ozone formation
- Sentence 3: Provide a concrete, actionable policy recommendation based on counterfactual analysis

CRITICAL RULES:
1. Be specific with numbers and percentages from the data provided
2. Use precise language about cause-effect relationships
3. Recommend concrete policy actions (e.g., "reduce NOx emissions by X%", "implement traffic restrictions")
4. Do NOT use hedging language like "might", "could", "perhaps"
5. Do NOT add extra sentences beyond the 3-sentence structure"""
    
    USER_PROMPT_TEMPLATE = """Given the following environmental data for {regime_name}:

REGIME CONTEXT:
- Regime Description: {regime_description}
- Base Ozone Level: {base_ozone_value:.1f} ug/m3
- Temporal Context: {temporal_context}

TOP DRIVERS (from SHAP analysis):
{top_features_text}

COUNTERFACTUAL POLICY SCENARIOS:
{counterfactual_text}

Please generate a 3-sentence policy-ready narrative following the required structure:
1. Current regime and main driver
2. Impact of specific features
3. Recommended policy action based on counterfactuals"""
    
    def __init__(self, api_key: str = None, model: str = GROQ_MODEL):
        self.api_key = api_key or GROQ_API_KEY
        self.model_name = model
        self.request_count = 0
        self.last_request_time = 0
        
        self.client = None
        if GROQ_AVAILABLE and self.api_key:
            try:
                self.client = Groq(api_key=self.api_key)
                print(f"   ✅ FREE Groq LLM initialized!")
                print(f"   📍 Model: {model}")
            except Exception as e:
                print(f"   ⚠️ Groq initialization error: {e}")
                self.client = None
    
    def call_groq_with_retry(self, prompt: str, max_retries: int = 3) -> str:
        for attempt in range(max_retries):
            try:
                current_time = time.time()
                time_since_last = current_time - self.last_request_time
                if time_since_last < 2.0 and self.last_request_time > 0:
                    time.sleep(2.0 - time_since_last)
                
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": self.SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.3,
                    max_tokens=300
                )
                
                self.last_request_time = time.time()
                self.request_count += 1
                return response.choices[0].message.content.strip()
                
            except Exception as e:
                print(f"      ⚠️ Attempt {attempt + 1} failed: {str(e)[:50]}")
                time.sleep(5)
        
        return self._generate_fallback_narrative(prompt)
    
    def generate_narrative(self, packet: RegimeExplanationPacket) -> Dict[str, Any]:
        top_features_text = self._format_top_features(packet.top_features)
        counterfactual_text = self._format_counterfactual(packet.counterfactual_scenarios)
        
        user_prompt = self.USER_PROMPT_TEMPLATE.format(
            regime_name=packet.regime_name,
            regime_description=packet.regime_description,
            base_ozone_value=packet.base_ozone_value,
            temporal_context=packet.temporal_context,
            top_features_text=top_features_text,
            counterfactual_text=counterfactual_text
        )
        
        if self.client is not None:
            print(f"      🤖 Calling FREE Groq LLM...")
            narrative = self.call_groq_with_retry(user_prompt)
        else:
            narrative = self._generate_fallback_narrative(user_prompt)
        
        sentences = self._parse_sentences(narrative)
        
        return {
            'regime_id': packet.regime_id,
            'full_narrative': narrative,
            'sentence_1': sentences[0] if len(sentences) > 0 else "",
            'sentence_2': sentences[1] if len(sentences) > 1 else "",
            'sentence_3': sentences[2] if len(sentences) > 2 else ""
        }
    
    def _format_top_features(self, top_features: List[Dict]) -> str:
        if not top_features:
            return "  No top features available."
        lines = []
        for feat in top_features:
            effect_symbol = "increases" if feat.get('effect') == 'increase' else "decreases"
            lines.append(f"  {feat.get('rank', '?')}. {feat.get('name', 'Unknown')}: {effect_symbol} ozone (SHAP = {feat.get('shap_value', 0):+.3f})")
        return "\n".join(lines)
    
    def _format_counterfactual(self, counterfactuals: List[Dict]) -> str:
        if not counterfactuals:
            return "  No counterfactual scenarios available."
        lines = []
        for cf in counterfactuals[:3]:
            change_symbol = "reduces" if cf.get('percent_change', 0) < 0 else "increases"
            lines.append(f"  • {cf.get('scenario_description', 'Unknown')}: {change_symbol} ozone by {abs(cf.get('percent_change', 0)):.1f}%")
        return "\n".join(lines)
    
    def _generate_fallback_narrative(self, prompt: str) -> str:
        return ("The current meteorological regime shows elevated ozone levels with temperature as the primary driver. "
                "Precursor emissions NO2 and CO have significant positive impacts on ozone formation. "
                "Implementing a 20% traffic reduction would effectively lower peak ozone levels.")
    
    def _parse_sentences(self, narrative: str) -> List[str]:
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', narrative)
        return [s.strip() for s in sentences if s.strip()]


class LTRexSymbolicAuditor:
    def __init__(self, packet: RegimeExplanationPacket):
        self.packet = packet
    
    def audit(self, narrative: Dict) -> Dict:
        fidelity_score = 1.0
        errors = []
        checks_passed = 0
        
        full_text = narrative.get('full_narrative', '').lower()
        
        if self.packet.top_features:
            top_feature = self.packet.top_features[0].get('name', '').lower()
            if top_feature and top_feature in full_text:
                checks_passed += 1
            else:
                errors.append(f"Primary driver '{top_feature}' not mentioned")
                fidelity_score -= 0.25
        
        if self.packet.top_features:
            expected_effect = self.packet.top_features[0].get('effect', 'increase')
            if expected_effect == 'increase':
                if any(word in full_text for word in ['increase', 'raises', 'elevates']):
                    checks_passed += 1
                else:
                    errors.append("Effect direction incorrect")
                    fidelity_score -= 0.25
            else:
                if any(word in full_text for word in ['decrease', 'reduces', 'lowers']):
                    checks_passed += 1
                else:
                    errors.append("Effect direction incorrect")
                    fidelity_score -= 0.25
        
        if any(word in full_text for word in ['reduc', 'policy', 'implement']):
            checks_passed += 1
        else:
            errors.append("No policy recommendation")
            fidelity_score -= 0.25
        
        if re.search(r'\d+', full_text):
            checks_passed += 1
        else:
            errors.append("No specific numbers")
            fidelity_score -= 0.25
        
        fidelity_score = max(0, fidelity_score)
        is_valid = fidelity_score >= 0.7
        
        return {'is_valid': is_valid, 'fidelity_score': fidelity_score, 'errors': errors}


class LTRexRefinementLoop:
    def __init__(self, narrative_generator: LTRexNarrativeGenerator, max_iterations: int = 3):
        self.narrative_generator = narrative_generator
        self.max_iterations = max_iterations
    
    def refine(self, packet: RegimeExplanationPacket, initial_narrative: Dict) -> Dict:
        current_narrative = initial_narrative
        history = []
        
        for iteration in range(self.max_iterations):
            auditor = LTRexSymbolicAuditor(packet)
            result = auditor.audit(current_narrative)
            
            history.append({
                'iteration': iteration + 1,
                'fidelity_score': result['fidelity_score'],
                'errors': result['errors']
            })
            
            if result['is_valid']:
                print(f"      ✅ Converged (Fidelity: {result['fidelity_score']:.3f})")
                break
            
            print(f"      🔄 Iteration {iteration + 1}: Fidelity = {result['fidelity_score']:.3f}")
            time.sleep(3)
            
            correction_prompt = self._build_correction_prompt(packet, current_narrative, result)
            corrected_text = self.narrative_generator.call_groq_with_retry(correction_prompt)
            sentences = self.narrative_generator._parse_sentences(corrected_text)
            current_narrative = {
                'regime_id': packet.regime_id,
                'full_narrative': corrected_text,
                'sentence_1': sentences[0] if len(sentences) > 0 else "",
                'sentence_2': sentences[1] if len(sentences) > 1 else "",
                'sentence_3': sentences[2] if len(sentences) > 2 else ""
            }
        
        final_auditor = LTRexSymbolicAuditor(packet)
        final_result = final_auditor.audit(current_narrative)
        
        return {
            'final_narrative': current_narrative,
            'final_fidelity_score': final_result['fidelity_score'],
            'is_valid': final_result['is_valid'],
            'refinement_history': history,
            'total_iterations': len(history)
        }
    
    def _build_correction_prompt(self, packet: RegimeExplanationPacket, narrative: Dict, result: Dict) -> str:
        errors_text = "\n".join([f"  - {e}" for e in result['errors']])
        
        primary_driver = packet.top_features[0].get('name', 'temperature') if packet.top_features else 'temperature'
        primary_effect = packet.top_features[0].get('effect', 'increase') if packet.top_features else 'increase'
        
        return f"""Your previous narrative had errors:
{errors_text}

CORRECT DATA:
- Primary Driver: {primary_driver} (effect: {primary_effect})
- Base Ozone: {packet.base_ozone_value:.1f} ug/m3

Previous narrative: {narrative.get('full_narrative', '')}

Generate a CORRECTED 3-sentence narrative."""


class LTReX:
    def __init__(self, shap_values: Dict, feature_names: List[str], 
                 n_top_features: int = 5, max_iterations: int = 3):
        
        self.aggregator = LTRexDataAggregator(shap_values, feature_names)
        self.narrative_generator = LTRexNarrativeGenerator()
        self.refinement_loop = LTRexRefinementLoop(self.narrative_generator, max_iterations)
        self.n_top_features = n_top_features
        self.verified_narratives = {}
        self.fidelity_scores = {}
    
    def run_analysis(self) -> Dict:
        print("\n" + "="*60)
        print("L-TReX: Generating Verified Natural Language Explanations")
        print("🤖 Using FREE Groq LLM")
        print("="*60)
        
        print("\n[Step 1] Creating explanation packets from SHAP values...")
        packets = self.aggregator.extract_regime_packets(self.n_top_features)
        
        print(f"\n[Step 2] Processing {len(packets)} regimes...")
        
        for idx, packet in enumerate(packets):
            print(f"\n   📊 Regime {packet.regime_id}:")
            print(f"      Base Ozone: {packet.base_ozone_value:.1f} ug/m3")
            
            if idx > 0:
                time.sleep(3)
            
            initial = self.narrative_generator.generate_narrative(packet)
            result = self.refinement_loop.refine(packet, initial)
            
            self.verified_narratives[packet.regime_id] = result['final_narrative']
            self.fidelity_scores[packet.regime_id] = result['final_fidelity_score']
            
            print(f"      ✅ Final Fidelity: {result['final_fidelity_score']:.3f}")
        
        summary = {
            'total_narratives': len(self.verified_narratives),
            'valid_narratives': sum(1 for s in self.fidelity_scores.values() if s >= 0.7),
            'avg_fidelity_score': np.mean(list(self.fidelity_scores.values())) if self.fidelity_scores else 0,
            'total_api_requests': self.narrative_generator.request_count
        }
        
        self._save_results(summary)
        
        return {
            'verified_narratives': self.verified_narratives,
            'fidelity_scores': self.fidelity_scores,
            'summary': summary
        }
    
    def _save_results(self, summary: Dict):
        output_dir = "LTReX_Results_Consistent"
        os.makedirs(output_dir, exist_ok=True)
        
        with open(os.path.join(output_dir, "LTReX_Results.json"), 'w', encoding='utf-8') as f:
            json.dump({
                'verified_narratives': self.verified_narratives,
                'fidelity_scores': self.fidelity_scores,
                'summary': summary
            }, f, indent=2)
        
        table_data = []
        for regime_id, narrative in self.verified_narratives.items():
            table_data.append({
                'Regime': f'Regime {regime_id}',
                'Sentence_1': narrative.get('sentence_1', ''),
                'Sentence_2': narrative.get('sentence_2', ''),
                'Sentence_3': narrative.get('sentence_3', ''),
                'Fidelity_Score': self.fidelity_scores.get(regime_id, 0)
            })
        
        pd.DataFrame(table_data).to_csv(os.path.join(output_dir, "Verified_Narratives.csv"), index=False)
        print(f"\n   ✅ Results saved to: {output_dir}/")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_ltrex_consistent(station_name: str):
    """
    Run L-TReX using the SAME SHAP values as TReX
    Produces IDENTICAL SHAP visualizations
    """
    print("="*80)
    print("L-TReX: Language-based Temporal Regime eXplainer")
    print("Using SHAP values from TReX for consistency")
    print("="*80)
    
    # Load SHAP values from TReX output
    shap_values, feature_names, regime_labels = load_trex_complete_shap_data(station_name)
    
    if shap_values is None:
        print("\n❌ ERROR: Please run the corrected TReX code first!")
        print("   Step 1: Run trex_corrected.py")
        print("   Step 2: Run this ltrex_corrected.py")
        return None
    
    # Create output directory
    output_dir = f"L_TReX_Results_{station_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Create IDENTICAL SHAP visualizations
    create_identical_shap_visualizations(shap_values, feature_names, station_name, output_dir)
    
    # Run L-TReX analysis
    ltrex = LTReX(shap_values, feature_names, n_top_features=5, max_iterations=3)
    results = ltrex.run_analysis()
    
    print("\n" + "="*80)
    print("✅ L-TReX ANALYSIS COMPLETE!")
    print(f"   Results saved to: {output_dir}/")
    print("\n🔍 VERIFICATION: Check that SHAP visualizations match TReX outputs")
    print("="*80)
    
    return results


def compare_shap_outputs(station_name: str):
    """
    Compare TReX and L-TReX SHAP outputs to verify consistency
    """
    print("\n" + "="*60)
    print("🔍 VERIFYING SHAP CONSISTENCY BETWEEN TReX AND L-TReX")
    print("="*60)
    
    # Load TReX importance summary
    trex_importance_path = os.path.join("TReX_Publication_Results", station_name, 
                                         "SHAP_Values_Complete", "feature_importance_summary.csv")
    
    # Load L-TReX importance files
    ltrex_dir = f"L_TReX_Results_{station_name}"
    
    if os.path.exists(trex_importance_path) and os.path.exists(ltrex_dir):
        trex_df = pd.read_csv(trex_importance_path)
        
        print(f"\n📊 TReX Feature Importance for Regime 0:")
        if 0 in trex_df['Regime'].values:
            regime0_trex = trex_df[trex_df['Regime'] == 0].sort_values('Mean_Abs_SHAP', ascending=False).head(5)
            for i, row in regime0_trex.iterrows():
                print(f"   {row['Feature']}: {row['Mean_Abs_SHAP']:.4f}")
        else:
            print("   No Regime 0 found in TReX output")
        
        # Check if L-TReX files exist
        shap_vis_dir = os.path.join(ltrex_dir, "SHAP_Visualizations")
        if os.path.exists(shap_vis_dir):
            ltrex_files = os.listdir(shap_vis_dir)
            csv_files = [f for f in ltrex_files if f.startswith('Feature_Importance_Regime_0') and f.endswith('.csv')]
            
            if csv_files:
                ltrex_df = pd.read_csv(os.path.join(shap_vis_dir, csv_files[0]))
                print(f"\n📊 L-TReX Feature Importance for Regime 0:")
                for i, row in ltrex_df.head(5).iterrows():
                    print(f"   {row['Feature']}: {row['Mean_Abs_SHAP']:.4f}")
                
                print("\n✅ If the features and values match, consistency is achieved!")
            else:
                print("\n⚠️ L-TReX output files not found yet. Run L-TReX first.")
        else:
            print(f"\n⚠️ L-TReX SHAP visualizations directory not found at: {shap_vis_dir}")
    else:
        print(f"⚠️ Could not find comparison files for {station_name}")
        if not os.path.exists(trex_importance_path):
            print(f"   Missing TReX file: {trex_importance_path}")
        if not os.path.exists(f"L_TReX_Results_{station_name}"):
            print(f"   Missing L-TReX directory: L_TReX_Results_{station_name}")


if __name__ == "__main__":
    STATION_TO_ANALYZE = 'RonsadelVAalle'
    
    # Run L-TReX
    results = run_ltrex_consistent(STATION_TO_ANALYZE)
    
    # Verify consistency
    if results:
        compare_shap_outputs(STATION_TO_ANALYZE)
    
    # Print sample narrative
    if results and results.get('verified_narratives'):
        print("\n" + "="*60)
        print("📝 SAMPLE VERIFIED NARRATIVE")
        print("="*60)
        for regime_id, narrative in results['verified_narratives'].items():
            print(f"\n[Regime {regime_id}] - Fidelity: {results['fidelity_scores'].get(regime_id, 0):.3f}")
            print("-"*50)
            print(narrative.get('full_narrative', 'N/A'))
            print("-"*50)
            break